In [1]:
import pandas as pd
import nltk
import re
import sklearn

nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer as wnl

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df = pd.read_json('/kaggle/input/datasets/haitranquangofficial/vietnamese-online-news-dataset/news_dataset.json')

In [3]:
df = df.head(5000).copy()

df.head()

,id,author,content,picture_count,processed,source,title,topic,url,crawled_at
0,218270,,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",3,0,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,2022-08-01 09:09:22.817308
1,218269,(Nguồn: Sina),"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",1,0,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,2022-08-01 09:09:21.181469
2,218268,Hồ Sỹ Anh,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,3,0,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,2022-08-01 09:09:15.311901
3,218267,Ngọc Ánh,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,1,0,vnexpress,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,2022-08-01 09:09:02.211498
4,218266,HẢI YẾN - MINH LÝ,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,12,0,soha,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,2022-08-01 09:09:01.601170


In [4]:
content_df = df[['content']].copy()
title_df = df[['title']].copy()

## Normalization

### Chuyển thành chữ viết thường

In [5]:
content_df['normalize'] = content_df['content'].str.lower()

content_df.head()

,content,normalize
0,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...","chiều 31/7, công an tỉnh thừa thiên - huế đã c..."
1,"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...","gần đây, thứ trưởng bộ phát triển kỹ thuật số,..."
2,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,kết quả thi tốt nghiệp thpt năm 2022 cho thấy ...
3,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,thống đốc kentucky andy beshear hôm 31/7 cho h...
4,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,vụ tai nạn giao thông liên hoàn trên phố đi bộ...


In [6]:
title_df['normalize'] = title_df['title'].str.lower()

title_df.head()

,title,normalize
0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...","tên cướp tiệm vàng tại huế là đại uý công an, ..."
1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G","bỏ qua mạng 5g, nga tiến thẳng từ 4g lên 6g"
2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,địa phương nào đứng đầu cả nước tổng điểm 3 mô...
3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,người chết trong mưa lũ 'nghìn năm có một' ở m...
4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...","hải phòng: hình ảnh xe ""điên"" gây tai nạn liên..."


### Xóa dấu câu

In [7]:
def remove_punctuation(text):
    text = re.sub(r'[^\w\s]',' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [8]:
content_df['normalize'] = content_df['normalize'].apply(remove_punctuation)

content_df.head

<bound method NDFrame.head of                                                 content  \
0     Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...   
1     Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...   
2     Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...   
3     Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...   
4     Vụ tai nạn giao thông liên hoàn trên phố đi bộ...   
...                                                 ...   
4995  Theo SCMP, một số đoạn đã hoàn thành của hầm n...   
4996  U18 nữ Việt Nam 1-0 Thái Lan. U18 nữ Việt Nam ...   
4997  Trong ba tuần, những tay đua của Tour de Franc...   
4998  Đại hội thể thao người khuyết tật Đông Nam Á (...   
4999  Tại hội thảo về thuế đối với thương mại điện t...   

                                              normalize  
0     chiều 31 7 công an tỉnh thừa thiên huế đã có t...  
1     gần đây thứ trưởng bộ phát triển kỹ thuật số t...  
2     kết quả thi tốt nghiệp thpt năm 2022 cho thấy ...  
3     thống đốc kentucky andy

In [9]:
title_df['normalize'] = title_df['normalize'].apply(remove_punctuation)

title_df.head()

,title,normalize
0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",tên cướp tiệm vàng tại huế là đại uý công an c...
1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",bỏ qua mạng 5g nga tiến thẳng từ 4g lên 6g
2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,địa phương nào đứng đầu cả nước tổng điểm 3 mô...
3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,người chết trong mưa lũ nghìn năm có một ở mỹ ...
4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",hải phòng hình ảnh xe điên gây tai nạn liên ho...


## Tokenize

In [10]:
content_df['tokens'] = content_df['normalize'].apply(word_tokenize)

content_df.head()

,content,normalize,tokens
0,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",chiều 31 7 công an tỉnh thừa thiên huế đã có t...,"[chiều, 31, 7, công, an, tỉnh, thừa, thiên, hu..."
1,"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",gần đây thứ trưởng bộ phát triển kỹ thuật số t...,"[gần, đây, thứ, trưởng, bộ, phát, triển, kỹ, t..."
2,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,kết quả thi tốt nghiệp thpt năm 2022 cho thấy ...,"[kết, quả, thi, tốt, nghiệp, thpt, năm, 2022, ..."
3,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,thống đốc kentucky andy beshear hôm 31 7 cho h...,"[thống, đốc, kentucky, andy, beshear, hôm, 31,..."
4,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,vụ tai nạn giao thông liên hoàn trên phố đi bộ...,"[vụ, tai, nạn, giao, thông, liên, hoàn, trên, ..."


In [11]:
title_df['tokens'] = title_df['normalize'].apply(word_tokenize)

title_df.head()

,title,normalize,tokens
0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",tên cướp tiệm vàng tại huế là đại uý công an c...,"[tên, cướp, tiệm, vàng, tại, huế, là, đại, uý,..."
1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",bỏ qua mạng 5g nga tiến thẳng từ 4g lên 6g,"[bỏ, qua, mạng, 5g, nga, tiến, thẳng, từ, 4g, ..."
2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,địa phương nào đứng đầu cả nước tổng điểm 3 mô...,"[địa, phương, nào, đứng, đầu, cả, nước, tổng, ..."
3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,người chết trong mưa lũ nghìn năm có một ở mỹ ...,"[người, chết, trong, mưa, lũ, nghìn, năm, có, ..."
4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",hải phòng hình ảnh xe điên gây tai nạn liên ho...,"[hải, phòng, hình, ảnh, xe, điên, gây, tai, nạ..."


## Stopwords

In [12]:
file_stopword = '/kaggle/input/datasets/minhkhoathi/vietnamese-stopwords/vietnamese-stopwords.txt'
with open (file_stopword, 'r', encoding='utf-8') as f:
    stop_words = set(line.strip() for line in f if line.strip())

title_df['tokens_no_stopwords'] = title_df['tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

content_df['tokens_no_stopwords'] = content_df['tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [13]:
title_df.head()

,title,normalize,tokens,tokens_no_stopwords
0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",tên cướp tiệm vàng tại huế là đại uý công an c...,"[tên, cướp, tiệm, vàng, tại, huế, là, đại, uý,...","[cướp, tiệm, vàng, huế, đại, uý, công, an, côn..."
1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",bỏ qua mạng 5g nga tiến thẳng từ 4g lên 6g,"[bỏ, qua, mạng, 5g, nga, tiến, thẳng, từ, 4g, ...","[mạng, 5g, nga, tiến, thẳng, 4g, 6g]"
2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,địa phương nào đứng đầu cả nước tổng điểm 3 mô...,"[địa, phương, nào, đứng, đầu, cả, nước, tổng, ...","[địa, phương, đứng, đầu, tổng, 3, môn, văn, to..."
3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,người chết trong mưa lũ nghìn năm có một ở mỹ ...,"[người, chết, trong, mưa, lũ, nghìn, năm, có, ...","[chết, mưa, lũ, nghìn, mỹ, 28]"
4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",hải phòng hình ảnh xe điên gây tai nạn liên ho...,"[hải, phòng, hình, ảnh, xe, điên, gây, tai, nạ...","[hải, phòng, hình, ảnh, xe, điên, tai, nạn, li..."


In [14]:
content_df.head()

,content,normalize,tokens,tokens_no_stopwords
0,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",chiều 31 7 công an tỉnh thừa thiên huế đã có t...,"[chiều, 31, 7, công, an, tỉnh, thừa, thiên, hu...","[chiều, 31, 7, công, an, tỉnh, thừa, thiên, hu..."
1,"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",gần đây thứ trưởng bộ phát triển kỹ thuật số t...,"[gần, đây, thứ, trưởng, bộ, phát, triển, kỹ, t...","[trưởng, phát, triển, kỹ, thuật, truyền, thông..."
2,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,kết quả thi tốt nghiệp thpt năm 2022 cho thấy ...,"[kết, quả, thi, tốt, nghiệp, thpt, năm, 2022, ...","[kết, thi, nghiệp, thpt, 2022, trung, bình, mô..."
3,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,thống đốc kentucky andy beshear hôm 31 7 cho h...,"[thống, đốc, kentucky, andy, beshear, hôm, 31,...","[thống, đốc, kentucky, andy, beshear, hôm, 31,..."
4,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,vụ tai nạn giao thông liên hoàn trên phố đi bộ...,"[vụ, tai, nạn, giao, thông, liên, hoàn, trên, ...","[vụ, tai, nạn, giao, thông, liên, hoàn, phố, đ..."


## Vectorization

In [15]:
title_df['final_text'] = title_df['tokens_no_stopwords'].apply(lambda tokens: ' '.join(tokens))

content_df['final_text'] = content_df['tokens_no_stopwords'].apply(lambda tokens: ' '.join(tokens))

In [16]:
title_df.head()

,title,normalize,tokens,tokens_no_stopwords,final_text
0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",tên cướp tiệm vàng tại huế là đại uý công an c...,"[tên, cướp, tiệm, vàng, tại, huế, là, đại, uý,...","[cướp, tiệm, vàng, huế, đại, uý, công, an, côn...",cướp tiệm vàng huế đại uý công an công tác trạ...
1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",bỏ qua mạng 5g nga tiến thẳng từ 4g lên 6g,"[bỏ, qua, mạng, 5g, nga, tiến, thẳng, từ, 4g, ...","[mạng, 5g, nga, tiến, thẳng, 4g, 6g]",mạng 5g nga tiến thẳng 4g 6g
2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,địa phương nào đứng đầu cả nước tổng điểm 3 mô...,"[địa, phương, nào, đứng, đầu, cả, nước, tổng, ...","[địa, phương, đứng, đầu, tổng, 3, môn, văn, to...",địa phương đứng đầu tổng 3 môn văn toán ngoại ngữ
3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,người chết trong mưa lũ nghìn năm có một ở mỹ ...,"[người, chết, trong, mưa, lũ, nghìn, năm, có, ...","[chết, mưa, lũ, nghìn, mỹ, 28]",chết mưa lũ nghìn mỹ 28
4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",hải phòng hình ảnh xe điên gây tai nạn liên ho...,"[hải, phòng, hình, ảnh, xe, điên, gây, tai, nạ...","[hải, phòng, hình, ảnh, xe, điên, tai, nạn, li...",hải phòng hình ảnh xe điên tai nạn liên hoàn p...


In [17]:
content_df.head()

,content,normalize,tokens,tokens_no_stopwords,final_text
0,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",chiều 31 7 công an tỉnh thừa thiên huế đã có t...,"[chiều, 31, 7, công, an, tỉnh, thừa, thiên, hu...","[chiều, 31, 7, công, an, tỉnh, thừa, thiên, hu...",chiều 31 7 công an tỉnh thừa thiên huế thông b...
1,"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",gần đây thứ trưởng bộ phát triển kỹ thuật số t...,"[gần, đây, thứ, trưởng, bộ, phát, triển, kỹ, t...","[trưởng, phát, triển, kỹ, thuật, truyền, thông...",trưởng phát triển kỹ thuật truyền thông truyền...
2,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,kết quả thi tốt nghiệp thpt năm 2022 cho thấy ...,"[kết, quả, thi, tốt, nghiệp, thpt, năm, 2022, ...","[kết, thi, nghiệp, thpt, 2022, trung, bình, mô...",kết thi nghiệp thpt 2022 trung bình môn toán n...
3,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,thống đốc kentucky andy beshear hôm 31 7 cho h...,"[thống, đốc, kentucky, andy, beshear, hôm, 31,...","[thống, đốc, kentucky, andy, beshear, hôm, 31,...",thống đốc kentucky andy beshear hôm 31 7 đợt m...
4,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,vụ tai nạn giao thông liên hoàn trên phố đi bộ...,"[vụ, tai, nạn, giao, thông, liên, hoàn, trên, ...","[vụ, tai, nạn, giao, thông, liên, hoàn, phố, đ...",vụ tai nạn giao thông liên hoàn phố đi tam bạc...


In [18]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

BOW = vectorizer.fit_transform(title_df['final_text'])

In [19]:
BOW.shape

(5000, 4183)

In [20]:
vectorizer.get_feature_names_out()

array(['00', '000', '0000', ..., 'ứng', '范明正总理视察太原钢铁厂二期扩建项目实施情况',
       '越南工业生产指数同比增长8'], dtype=object)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(title_df['final_text'])

In [22]:
X.shape

(5000, 4183)

## Stemming

In [23]:
from nltk.stem import *
text = 'play playing played plays hourse hourses am is are be'
tokens = word_tokenize(text)
tokens


['play',
 'playing',
 'played',
 'plays',
 'hourse',
 'hourses',
 'am',
 'is',
 'are',
 'be']

In [24]:
stemmer = PorterStemmer()

singles = [stemmer.stem(word) for word in tokens]

singles

['play', 'play', 'play', 'play', 'hours', 'hours', 'am', 'is', 'are', 'be']

In [25]:
final_text = ' '.join(singles)
final_text

'play play play play hours hours am is are be'

## Lemmatization

In [26]:
nltk.download('wordnet')
nltk.download('omw-1.4')


text = 'play playing played plays am is are be'
tokens = word_tokenize(text)

Lemmatizer = WordNetLemmatizer()
for word in tokens:
    Lemmatized = Lemmatizer.lemmatize(word, pos = 'v')
    print(Lemmatized)

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


play
play
play
play
be
be
be
be
